# Test Notebook for "Music CurAItor" Project  

For testing purposes **ONLY**  
WARNING: Some sections of the code may broken/non-functional

In [ ]:
import os
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

api_key = os.getenv("GEMINI_API_KEY") # change accordingly

# Mock data
songs = [
        {"title": "It Ain't Like That", "artist": "Alice in Chains", "genre": "grunge"},
        {"title": "Interstate Love Song", "artist": "Stone Temple Pilots", "genre": "grunge"},
        {"title": "Down in a Hole", "artist": "Alice in Chains", "genre": "grunge"},
        {"title": "Blue in Green", "artist": "Miles Davis", "genre": "jazz"},
        {"title": "Parisienne Walkway", "artist": "Gary Moore", "genre": "rock"},
        {"title": "Say You Love Me", "artist": "Fleetwood Mac", "genre": "rock"},
        {"title": "Subterranean Homesick Blues", "artist": "Bob Dylan", "genre": "rock"},
        {"title": "Don't Think Twice, It's Alright", "artist": "Bob Dylan", "genre": "folk"},
        {"title": "Cannock Chase", "artist": "Labi Siffre", "genre": "folk"},
    ]

# Helper function for get_songs_by_genre tool
def find_songs_by_genre(genre):
    return [song for song in songs if song["genre"] == genre]

In [ ]:
# Sample tool
@tool(description="Get the current weather in a given location")
def get_weather(location: str) -> str:
    return "It's sunny."

# My tool
@tool(description="Get songs from a specific genre")
def get_songs_by_genre(genre: str) -> str:

    results = find_songs_by_genre(genre)

    if not results:
        return "No songs found."
    return "\n".join(
        [f"{song['title']} by {song['artist']}" for song in results]
    )

In [3]:
# Initialize LLM with 3.1 Pro (for tools use) and bind required tools
model_with_tools = ChatGoogleGenerativeAI(
    model="gemini-3.1-pro-preview",
    temperature=0.6,
    google_api_key=api_key
    ).bind_tools([get_songs_by_genre])

# Step 1: Model generates tool calls
messages = [HumanMessage("Give me some rock songs to listen to.")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Check the tool calls in the response
print(ai_msg.tool_calls)

# NEW STEP 2:
tools = {
    "get_weather": get_weather,
    "get_songs_by_genre": get_songs_by_genre
}

for tool_call in ai_msg.tool_calls:
    tool_name = tool_call["name"]
    selected_tool = tools[tool_name]

    tool_result = selected_tool.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

[{'name': 'get_songs_by_genre', 'args': {'genre': 'rock'}, 'id': 'f924f5f7-9ecc-4f56-8b3c-5ad3a2bacced', 'type': 'tool_call'}]


In [14]:
print(final_response.content[0]['text'])

Here are some great rock songs for you to listen to:

* **Parisienne Walkway** by Gary Moore
* **Say You Love Me** by Fleetwood Mac
* **Subterranean Homesick Blues** by Bob Dylan

Happy listening! Let me know if you'd like more recommendations.


## Chatbot section

In [15]:
messages = []

while True:

    user_input = input("\nYou: ")

    if user_input.lower() in ["quit", "exit"]:
        break

    messages.append(HumanMessage(user_input))

    ai_msg = model_with_tools.invoke(messages)
    messages.append(ai_msg)

    print(ai_msg.tool_calls)

    for tool_call in ai_msg.tool_calls:
        tool_name = tool_call["name"]
        selected_tool = tools[tool_name]

        tool_result = selected_tool.invoke(tool_call)
        messages.append(tool_result)

    final_response = model_with_tools.invoke(messages)
    messages.append(final_response)

    print("\nAgent:", final_response.content[0]["text"])

[{'name': 'get_songs_by_genre', 'args': {'genre': 'grunge'}, 'id': '9037e76c-a379-43ea-822d-c23f76da71d8', 'type': 'tool_call'}]

Agent: Here are some great grunge songs for you to listen to:

* **"It Ain't Like That"** by Alice in Chains
* **"Interstate Love Song"** by Stone Temple Pilots
* **"Down in a Hole"** by Alice in Chains

Happy listening! Let me know if you'd like more recommendations.
[{'name': 'get_songs_by_genre', 'args': {'genre': 'rock'}, 'id': '3a01b28d-d9cb-4579-be52-71d9ee7dd4d1', 'type': 'tool_call'}]

Agent: Here are some excellent rock songs for you to check out:

* **"Parisienne Walkways"** by Gary Moore
* **"Say You Love Me"** by Fleetwood Mac
* **"Subterranean Homesick Blues"** by Bob Dylan

Enjoy the music! Let me know if you want more suggestions or perhaps a different sub-genre of rock.
[{'name': 'get_songs_by_genre', 'args': {'genre': 'classic rock'}, 'id': 'ec710b78-75b1-4403-a537-85ee07ba2953', 'type': 'tool_call'}, {'name': 'get_songs_by_genre', 'args': {